# 01 — Radiological concept audit (CLEAR)

**Diagnostic only. A concept audit is not expected to change AUROC.**

CLEAR is Apache-2.0 (`peterhan91/CLEAR`, code at `github.com/peterhan91/CLEAR`),
DINOv2 ViT-B/14 with registers, **448x448** input, 768-d shared space.

**This notebook refuses to compute concept scores unless** images are attached AND
`manifest.json` records that the image licence permits Kaggle use. If either is
missing it runs preflight only and reports the blocker. It never fabricates scores.

Concepts are **model-derived proxies**, not ground truth, and `past_tb` is a
clinical-history variable — image-visible fibrosis must not be inferred from it.


In [ ]:
import os, sys, json, time, hashlib, platform
from pathlib import Path
WORK = Path('/kaggle/working'); INP = Path('/kaggle/input')
CODE = next((p for p in INP.glob('*/src/guards.py')), None)
if CODE is None:
    raise SystemExit('attach the code dataset; see KAGGLE_INPUT_CHECKLIST.md')
sys.path.insert(0, str(CODE.parent))
import guards as G
print('code from', CODE.parent)


In [ ]:
# --- the concept list is FIXED BEFORE any association is computed
CONCEPTS = ['fibrosis', 'scarring', 'volume loss', 'calcified granuloma',
            'calcification', 'cavity', 'focal opacity', 'consolidation',
            'pleural thickening', 'nodule', 'reticular opacity', 'bronchiectasis']
print('%d preregistered concepts' % len(CONCEPTS))
print('Each must be mapped to an exact phrase in the official CLEAR concept bank.')
print('Any concept absent from the released vocabulary is DROPPED and recorded,')
print('never substituted with an invented phrase.')


In [ ]:
# --- gate: images + licence, or stop
import json
from pathlib import Path
man = next((p for p in INP.glob('*/manifest.json')), None)
lic_ok = False
if man:
    M = json.loads(Path(man).read_text())
    lic_ok = str(M.get('licences', {}).get('vietnam_images', {})
                  .get('kaggle_use_permitted', 'unknown')).lower() == 'yes'
d = next((p for p in INP.rglob('vietnam_img512') if p.is_dir()), None)
n_img = len(list(d.glob('*.png'))) if d else 0
BLOCKED = (not lic_ok) or n_img == 0
print('image licence cleared for Kaggle:', lic_ok)
print('images attached:', n_img)
if BLOCKED:
    print()
    print('STAGE 1 CONCEPT INFERENCE IS BLOCKED.')
    print('Unknown licence == blocked. No concept score will be computed or')
    print('invented. The score-only subgroup analysis below still runs if the')
    print('frozen per-case scores and metadata are attached.')


In [ ]:
# --- score-only subgroup analysis (TUNE ONLY; REPORT is never read)
import numpy as np, pandas as pd
sys.path.insert(0, str(CODE.parent))
import evaluate as E
sc = next((p for p in INP.rglob('vietnam_case_scores.csv')), None)
mt = next((p for p in INP.rglob('Lunit_Versions_-_Dryad_Data_Set.csv')), None)
sp = next((p for p in INP.rglob('vietnam_tune_report_split.json')), None)
if not (sc and mt and sp):
    print('score/metadata/split inputs missing - nothing to analyse')
else:
    S = pd.read_csv(sc); Mt = pd.read_csv(mt).set_index('pid')
    tune = set(str(x) for x in json.loads(Path(sp).read_text())['tune'])
    S['pid'] = S['pid'].astype(str)
    S = S[S.pid.isin(tune)].copy()
    assert len(S) == len(set(S.pid)), 'duplicate pid'
    S['y'] = [int(Mt.loc[int(p), 'xpert_result']) for p in S.pid]
    S['past_tb'] = [int(Mt.loc[int(p), 'past_tb']) for p in S.pid]
    rows = []
    for yv in (0, 1):
        for pv in (0, 1):
            g = S[(S.y == yv) & (S.past_tb == pv)]
            rows.append({'xpert': yv, 'past_tb': pv, 'n': len(g),
                         'median_score': float(np.median(g.score_baseline))})
    out = pd.DataFrame(rows); print(out.to_string(index=False))
    out.to_csv(Path('%s') / 'tables_subgroup_metrics.csv' , index=False)


In [ ]:
# --- concept inference, only if unblocked
if BLOCKED:
    print('skipped: see the gate above')
else:
    print('Load CLEAR from the attached weights or, with internet enabled,')
    print('from the pinned HF revision. Preserve the OFFICIAL 448px preprocessing.')
    print('Then: encode TUNE images -> project to the concept bank -> write')
    print('tables/concept_score_summary.csv and')
    print('tables/concept_model_score_associations.csv with bootstrap CIs and')
    print('Benjamini-Hochberg correction across the %d concepts.' % len(CONCEPTS))
    print()
    print('Associations are reported stratified by active-TB label AND past-TB')
    print('history, including among POSITIVES - a concept that also marks true')
    print('active disease must not be suppressed.')
